In [ ]:
# Update pip to latest version
!pip install --upgrade pip

# Install latest boto3 and dependencies
!pip install --upgrade boto3 requests

# AWS Bedrock Agent Core Gateway Setup with Cedar Policies

This notebook demonstrates how to set up an **AWS Bedrock Agent Core Gateway** with policy enforcement using Cedar policies. The gateway enables secure access to Lambda functions through an MCP (Model Context Protocol) interface with OAuth-based authentication and fine-grained access control.

## Overview

This setup includes:
- **IAM Roles & Policies**: Execution and Lambda invocation permissions
- **Cognito User Pool**: OAuth authentication for gateway access
- **Gateway Configuration**: MCP protocol with JWT authentication
- **Policy Engine**: Cedar-based authorization with permit/forbid rules
- **Gateway Targets**: Lambda function targets with tool schemas
- **Access Control**: Attribute-based policies enforcing conditional access

---

## Prerequisites

- AWS Account with Bedrock Agent Core access
- Python 3.12+
- Boto3 and requests libraries installed

In [2]:
import boto3
import requests
import json
import time
from datetime import datetime

# Verify library versions
print(f"boto3: {boto3.__version__}")
print(f"requests: {requests.__version__}")

# Generate unique identifier for this run using formatted timestamp
now = datetime.now()
TIMESTAMP = now.strftime('%d%m%Y%H%M%S')  # Format: DDMMYYYYHHMMSS
print(f"Run timestamp: {TIMESTAMP}")

boto3: 1.42.62
requests: 2.32.5
Run timestamp: 05032026172451


## Section 1: Initialize AWS Clients and Generate Identifiers

This section initializes the AWS clients for IAM, Bedrock Agent Core, Lambda, and Cognito. A unique timestamp is generated to ensure all created resources have unique names.

## Section 2: Create IAM Roles

### 2.1 Gateway Execution Role

Create an IAM role that will be assumed by the Bedrock Agent Core Gateway. This role has permissions for:
- Policy engine operations (get, create, authorize, evaluate)
- Lambda function invocation
- IAM role pass-through

In [3]:
iam = boto3.client('iam', region_name='us-east-1')

# Define trust policy for the gateway role
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": [
                    "bedrock-agentcore.amazonaws.com",
                    "bedrock.amazonaws.com"
                ]
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

role_name = f'gateway-execution-role-{TIMESTAMP}'

# Create the gateway execution role
response = iam.create_role(
    RoleName=role_name,
    AssumeRolePolicyDocument=json.dumps(trust_policy)
)
gateway_role_arn = response['Role']['Arn']
print("✓ Gateway execution role created successfully")

✓ Gateway execution role created successfully


### 2.2 Attach Comprehensive Permissions to Gateway Role

Attach a policy with permissions for:
- **Policy Engine Access**: Get policy engine, get gateway, create gateway, authorize actions, evaluate policies
- **Lambda Invocation**: Invoke Lambda functions
- **PassRole**: Pass the gateway role to services

In [4]:
# Define comprehensive permissions policy for the gateway
comprehensive_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "PolicyEngineAccess",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetPolicyEngine",
                "bedrock-agentcore:GetGateway",
                "bedrock-agentcore:CreateGateway",
                "bedrock-agentcore:AuthorizeAction",
                "bedrock-agentcore:PartiallyAuthorizeActions",
                "bedrock-agentcore:Evaluate"
            ],
            "Resource": "*"
        },
        {
            "Sid": "LambdaInvoke",
            "Effect": "Allow",
            "Action": "lambda:InvokeFunction",
            "Resource": "*"
        },
        {
            "Sid": "PassRole",
            "Effect": "Allow",
            "Action": "iam:PassRole",
            "Resource": gateway_role_arn
        }
    ]
}

# Attach the policy to the gateway role
try:
    iam.put_role_policy(
        RoleName=role_name,
        PolicyName='ComprehensiveGatewayPermissions',
        PolicyDocument=json.dumps(comprehensive_policy)
    )
    print("✓ Comprehensive permissions policy attached to gateway role")
except Exception as e:
    print(f"Error attaching policy: {e}")
    raise

✓ Comprehensive permissions policy attached to gateway role


### 2.3 Create Lambda Execution Role

Create a separate IAM role for Lambda functions with basic execution permissions for CloudWatch Logs.

In [5]:
# Define trust policy for Lambda
lambda_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }
    ]
}

lambda_role_name = f'lambda-gateway-target-role-{TIMESTAMP}'

# Create the Lambda execution role
response = iam.create_role(
    RoleName=lambda_role_name,
    AssumeRolePolicyDocument=json.dumps(lambda_trust_policy)
)
lambda_role_arn = response['Role']['Arn']

# Attach AWS managed execution policy for Lambda
iam.attach_role_policy(
    RoleName=lambda_role_name,
    PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
)
print("✓ Lambda execution role created and configured")

✓ Lambda execution role created and configured


## Section 3: Create Policy Engine

The Cedar Policy Engine is used to enforce access control policies on the gateway. It will store permit and forbid rules to determine which actions are allowed.

In [6]:
agentcore = boto3.client('bedrock-agentcore-control', region_name='us-east-1')

policy_engine_name = f'GatewayPolicyEngine_{TIMESTAMP}'

# Create the policy engine
response = agentcore.create_policy_engine(
    name=policy_engine_name
)
policy_engine_id = response['policyEngineId']
print("✓ Policy Engine created successfully")

✓ Policy Engine created successfully


## Section 4: Create Lambda Functions

Create Lambda functions that will serve as targets for the gateway. These functions will be invoked via the MCP protocol when tools are called.

In [7]:
lambda_client = boto3.client('lambda', region_name='us-east-1')

# Define basic Lambda function code
lambda_code = '''
def lambda_handler(event, context):
    return {
        "statusCode": 200,
        "body": {"message": "Gateway test successful", "event": event}
    }
'''

# Package the Lambda function code into a ZIP file
import zipfile
import io

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zip_file:
    zip_file.writestr('lambda_function.py', lambda_code)

lambda_function_name = f'gateway-target-function-{TIMESTAMP}'

# Wait for IAM role propagation
print("Waiting for Lambda role to propagate...")
time.sleep(5)

# Create the Lambda function
response = lambda_client.create_function(
    FunctionName=lambda_function_name,
    Runtime='python3.12',
    Role=lambda_role_arn,
    Handler='lambda_function.lambda_handler',
    Code={'ZipFile': zip_buffer.getvalue()}
)
lambda_arn = response['FunctionArn']
print("✓ Lambda function created successfully")

Waiting for Lambda role to propagate...


✓ Lambda function created successfully


## Section 5: Set Up OAuth Authentication with Cognito

Create a Cognito User Pool with a test user for OAuth authentication. The gateway will use JWT tokens from this pool to authenticate requests.

In [8]:
cognito = boto3.client('cognito-idp', region_name='us-east-1')

# Create a Cognito User Pool for gateway authentication
user_pool_name = f'gateway-user-pool-{TIMESTAMP}'
pool_response = cognito.create_user_pool(
    PoolName=user_pool_name,
    Policies={
        'PasswordPolicy': {
            'MinimumLength': 8,
            'RequireUppercase': True,
            'RequireLowercase': True,
            'RequireNumbers': True,
            'RequireSymbols': False
        }
    }
)
user_pool_id = pool_response['UserPool']['Id']
print("✓ Cognito User Pool created successfully")

# Define client configuration variables
cognito_client_name = f'gateway-client-{TIMESTAMP}'
test_username = 'gateway_test_user'

✓ Cognito User Pool created successfully


### 5.1 Create Cognito Client and Test User

Set up the Cognito client with auth flows enabled and create a test user for demonstration purposes.

In [9]:
# Create Cognito User Pool Client with OAuth auth flows enabled
client_response = cognito.create_user_pool_client(
    UserPoolId=user_pool_id,
    ClientName=cognito_client_name,
    ExplicitAuthFlows=[
        'ALLOW_ADMIN_USER_PASSWORD_AUTH', 
        'ALLOW_REFRESH_TOKEN_AUTH',
        'ALLOW_USER_PASSWORD_AUTH'
    ],
    GenerateSecret=False
)
client_id = client_response['UserPoolClient']['ClientId']

# Create a test user for authentication testing
try:
    cognito.admin_create_user(
        UserPoolId=user_pool_id,
        Username=test_username,
        TemporaryPassword='TempPass123!',
        MessageAction='SUPPRESS'
    )
    print("✓ Test user created")
except cognito.exceptions.UsernameExistsException:
    print("✓ Test user already exists")

# Set permanent password for the test user
cognito.admin_set_user_password(
    UserPoolId=user_pool_id,
    Username=test_username,
    Password='TestPassword123!',
    Permanent=True
)
print("✓ Cognito client and test user configured successfully")

✓ Test user created


✓ Cognito client and test user configured successfully


### 5.2 Verify Token Structure

Inspect the JWT token claims to ensure proper configuration for gateway authentication.

In [10]:
# Helper function to decode JWT tokens
def decode_token(token):
    """Decode JWT token payload without signature verification"""
    import base64
    parts = token.split('.')
    if len(parts) != 3:
        return None
    
    payload = parts[1]
    # Add padding if needed
    padding = 4 - len(payload) % 4
    if padding != 4:
        payload += '=' * padding
    
    try:
        decoded = base64.urlsafe_b64decode(payload)
        return json.loads(decoded)
    except:
        return None

# Obtain authentication tokens
auth_response = cognito.admin_initiate_auth(
    UserPoolId=user_pool_id,
    ClientId=client_id,
    AuthFlow='ADMIN_NO_SRP_AUTH',
    AuthParameters={
        'USERNAME': test_username,
        'PASSWORD': 'TestPassword123!'
    }
)

# Extract tokens
access_token = auth_response['AuthenticationResult']['AccessToken']
id_token = auth_response['AuthenticationResult']['IdToken']

# Display token claims
print("Access Token Claims:")
access_claims = decode_token(access_token)
print(f"  - Token Use: {access_claims.get('token_use')}")
print(f"  - Client ID: {access_claims.get('client_id')}")
print(f"  - Scope: {access_claims.get('scope')}")

print("\nID Token Claims:")
id_claims = decode_token(id_token)
print(f"  - Token Use: {id_claims.get('token_use')}")
print(f"  - Audience: {id_claims.get('aud')}")
print(f"  - Username: {id_claims.get('cognito:username')}")

print("\n✓ Token validation successful")

Access Token Claims:
  - Token Use: access
  - Client ID: 6ui59fefc1de313tece9dt0vfm
  - Scope: aws.cognito.signin.user.admin

ID Token Claims:
  - Token Use: id
  - Audience: 6ui59fefc1de313tece9dt0vfm
  - Username: gateway_test_user

✓ Token validation successful


## Section 6: Create and Configure Gateway

Create the primary Bedrock Agent Core Gateway with:
- Custom JWT authentication via Cognito
- Policy engine in ENFORCE mode for authorization
- MCP protocol for communication

In [11]:
# Retrieve account ID (needed for policy engine ARN)
sts = boto3.client('sts', region_name='us-east-1')
account_id = sts.get_caller_identity()['Account']

# Construct the policy engine ARN
policy_engine_arn = f"arn:aws:bedrock-agentcore:us-east-1:{account_id}:policy-engine/{policy_engine_id}"

# OpenID Discovery URL for Cognito
discovery_url = f"https://cognito-idp.us-east-1.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

gateway_name = f'gateway-with-policy-enforce-{TIMESTAMP}'

# Create the gateway with policy engine configuration
gateway_response = agentcore.create_gateway(
    name=gateway_name,
    roleArn=gateway_role_arn,
    protocolType='MCP',
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration={
        'customJWTAuthorizer': {
            'discoveryUrl': discovery_url,
            'allowedAudience': [client_id]
        }
    },
    policyEngineConfiguration={
        'arn': policy_engine_arn,
        'mode': 'ENFORCE'
    }
)
gateway_id = gateway_response['gatewayId']
gateway_url = gateway_response['gatewayUrl']

# Ensure URL has /mcp endpoint
gateway_url_full = gateway_url if gateway_url.endswith('/mcp') else f"{gateway_url}/mcp"
print("✓ Gateway created successfully with policy enforcement enabled")

✓ Gateway created successfully with policy enforcement enabled


## Section 7: Create Gateway Target and Policies

### 7.1 Create Gateway Target

Add a Lambda function as a gateway target with a tool schema defining the available operations.

In [12]:
target_id = f'lambda-target-{TIMESTAMP}'

# Create a gateway target for the Lambda function
target_response = agentcore.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name=target_id,
    targetConfiguration={
        'mcp': {
            'lambda': {
                'lambdaArn': lambda_arn,
                'toolSchema': {
                    'inlinePayload': [
                        {
                            'name': 'test_tool',
                            'description': 'A test tool that demonstrates the gateway functionality',
                            'inputSchema': {
                                'type': 'object',
                                'properties': {
                                    'message': {
                                        'type': 'string',
                                        'description': 'A test message'
                                    }
                                },
                                'required': []
                            }
                        }
                    ]
                }
            }
        }
    },
    credentialProviderConfigurations=[
        {
            'credentialProviderType': 'GATEWAY_IAM_ROLE'
        }
    ]
)
print("✓ Gateway target created successfully")

# Add Lambda invoke permission to the gateway role
lambda_invoke_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "lambda:InvokeFunction",
            "Resource": lambda_arn
        }
    ]
}

iam.put_role_policy(
    RoleName=role_name,
    PolicyName='GatewayLambdaInvokePolicy',
    PolicyDocument=json.dumps(lambda_invoke_policy)
)
print("✓ Lambda invoke permissions configured for gateway role")

✓ Gateway target created successfully


✓ Lambda invoke permissions configured for gateway role


### 7.2 Create Cedar Permit Policy

Define a Cedar permit policy that allows OAuth users to access the target and its tools.

In [13]:
# Define Cedar policy to permit OAuth users to access the target
cedar_policy = f"""permit (
  principal is AgentCore::OAuthUser,
  action in
    [AgentCore::Action::"{target_id}",
     AgentCore::Action::"{target_id}___test_tool"],
  resource ==
    AgentCore::Gateway::"arn:aws:bedrock-agentcore:us-east-1:{account_id}:gateway/{gateway_id}"
);"""

policy_name = f'oauth_user_policy_{TIMESTAMP}'

# Create the permit policy in the policy engine
policy_response = agentcore.create_policy(
    policyEngineId=policy_engine_id,
    name=policy_name,
    definition={'cedar': {'statement': cedar_policy}}
)
policy_id = policy_response['policyId']
print("✓ Cedar permit policy created successfully")

✓ Cedar permit policy created successfully


## Section 8: Test Basic Gateway Invocation

Test the gateway by calling the tools/list endpoint to retrieve available tools from the target.

In [14]:
# Obtain a fresh ID token for authentication
auth_response = cognito.admin_initiate_auth(
    UserPoolId=user_pool_id,
    ClientId=client_id,
    AuthFlow='ADMIN_NO_SRP_AUTH',
    AuthParameters={
        'USERNAME': test_username,
        'PASSWORD': 'TestPassword123!'
    }
)

id_token = auth_response['AuthenticationResult']['IdToken']

# Prepare headers with Bearer token
headers = {
    'Authorization': f'Bearer {id_token}',
    'Content-Type': 'application/json'
}

# Prepare JSON-RPC request to list available tools
test_payload = {
    "jsonrpc": "2.0",
    "method": "tools/list",
    "params": {},
    "id": 1
}

# Invoke the gateway
print("Testing gateway with tools/list endpoint...")
response = requests.post(
    gateway_url_full,
    json=test_payload,
    headers=headers,
    timeout=30
)

print(f"Response Status: {response.status_code}")

if response.status_code == 200:
    print("✓ Gateway invocation successful!")
    response_json = response.json()
    if 'result' in response_json:
        tools = response_json['result'].get('tools', [])
        print(f"  Available tools: {len(tools)}")
        for tool in tools:
            print(f"    - {tool['name']}")
else:
    print(f"✗ Gateway invocation failed with status {response.status_code}")
    raise RuntimeError(f"Gateway invocation failed: {response.text}")

Testing gateway with tools/list endpoint...


Response Status: 200
✓ Gateway invocation successful!
  Available tools: 1
    - lambda-target-05032026172451___test_tool


## Section 9: Create Advanced Demonstration Setup

### 9.1 Set Up New Infrastructure with Integer Parameter Validation

Create a second gateway and target to demonstrate parameter-based access control using forbid policies.

In [15]:
# Create a new target with integer parameter schema
new_target_id = f'lambda-target-with-int-{TIMESTAMP}'

new_target_response = agentcore.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name=new_target_id,
    targetConfiguration={
        'mcp': {
            'lambda': {
                'lambdaArn': lambda_arn,
                'toolSchema': {
                    'inlinePayload': [
                        {
                            'name': 'process_number',
                            'description': 'A tool that processes an integer number with conditional logic',
                            'inputSchema': {
                                'type': 'object',
                                'properties': {
                                    'message': {
                                        'type': 'string',
                                        'description': 'A descriptive message'
                                    },
                                    'count': {
                                        'type': 'integer',
                                        'description': 'A numeric count value'
                                    }
                                },
                                'required': ['count']
                            }
                        }
                    ]
                }
            }
        }
    },
    credentialProviderConfigurations=[
        {
            'credentialProviderType': 'GATEWAY_IAM_ROLE'
        }
    ]
)
print("✓ New target with integer schema created successfully")

✓ New target with integer schema created successfully


### 9.2 Create Secondary Policy Engine and Gateway

Set up a separate policy engine and gateway for demonstrating conditional access control.

In [16]:
import uuid

# Create a new policy engine with unique identifier
new_policy_engine_name = f'PE_{TIMESTAMP}_{str(uuid.uuid4())[:4]}'

try:
    new_policy_engine_response = agentcore.create_policy_engine(
        name=new_policy_engine_name
    )
    new_policy_engine_id = new_policy_engine_response['policyEngineId']
except agentcore.exceptions.ConflictException:
    # Retry with longer unique suffix if name conflicts
    new_policy_engine_name = f'PE_{TIMESTAMP}_{str(uuid.uuid4())[:8]}'
    new_policy_engine_response = agentcore.create_policy_engine(
        name=new_policy_engine_name
    )
    new_policy_engine_id = new_policy_engine_response['policyEngineId']

print("✓ New policy engine created")

# Wait for policy engine to become ACTIVE
print("Waiting for policy engine to be ready...")
max_attempts = 30
for attempt in range(max_attempts):
    try:
        pe_response = agentcore.get_policy_engine(policyEngineId=new_policy_engine_id)
        if pe_response.get('status') == 'ACTIVE':
            break
    except:
        pass
    
    if attempt < max_attempts - 1:
        time.sleep(2)

# Create new gateway with unique suffix
new_gateway_name = f'gateway-int-target-{TIMESTAMP}-{str(uuid.uuid4())[:4]}'

try:
    new_gateway_response = agentcore.create_gateway(
        name=new_gateway_name,
        roleArn=gateway_role_arn,
        protocolType='MCP',
        authorizerType='CUSTOM_JWT',
        authorizerConfiguration={
            'customJWTAuthorizer': {
                'discoveryUrl': discovery_url,
                'allowedAudience': [client_id]
            }
        },
        policyEngineConfiguration={
            'arn': f"arn:aws:bedrock-agentcore:us-east-1:{account_id}:policy-engine/{new_policy_engine_id}",
            'mode': 'ENFORCE'
        }
    )
    new_gateway_id = new_gateway_response['gatewayId']
    new_gateway_url = new_gateway_response['gatewayUrl']
except agentcore.exceptions.ConflictException:
    # Retry with longer unique suffix if name conflicts
    new_gateway_name = f'gateway-int-target-{TIMESTAMP}-{str(uuid.uuid4())[:8]}'
    new_gateway_response = agentcore.create_gateway(
        name=new_gateway_name,
        roleArn=gateway_role_arn,
        protocolType='MCP',
        authorizerType='CUSTOM_JWT',
        authorizerConfiguration={
            'customJWTAuthorizer': {
                'discoveryUrl': discovery_url,
                'allowedAudience': [client_id]
            }
        },
        policyEngineConfiguration={
            'arn': f"arn:aws:bedrock-agentcore:us-east-1:{account_id}:policy-engine/{new_policy_engine_id}",
            'mode': 'ENFORCE'
        }
    )
    new_gateway_id = new_gateway_response['gatewayId']
    new_gateway_url = new_gateway_response['gatewayUrl']

print("✓ New gateway created")

# Wait for gateway to become ACTIVE before creating targets
print("Waiting for gateway to be ready...")
max_attempts = 30
for attempt in range(max_attempts):
    try:
        gw_response = agentcore.get_gateway(gatewayIdentifier=new_gateway_id)
        if gw_response.get('status') == 'ACTIVE':
            break
    except:
        pass
    
    if attempt < max_attempts - 1:
        time.sleep(2)

# Create gateway target for the new integer-based target
new_target_gateway_response = agentcore.create_gateway_target(
    gatewayIdentifier=new_gateway_id,
    name=new_target_id,
    targetConfiguration={
        'mcp': {
            'lambda': {
                'lambdaArn': lambda_arn,
                'toolSchema': {
                    'inlinePayload': [
                        {
                            'name': 'process_number',
                            'description': 'A tool that processes an integer number with conditional logic',
                            'inputSchema': {
                                'type': 'object',
                                'properties': {
                                    'message': {
                                        'type': 'string',
                                        'description': 'A descriptive message'
                                    },
                                    'count': {
                                        'type': 'integer',
                                        'description': 'A numeric count value'
                                    }
                                },
                                'required': ['count']
                            }
                        }
                    ]
                }
            }
        }
    },
    credentialProviderConfigurations=[
        {
            'credentialProviderType': 'GATEWAY_IAM_ROLE'
        }
    ]
)
print("✓ Gateway target created for new gateway")

# Create Cedar permit policy for the new gateway
cedar_policy_int = f"""permit (
  principal is AgentCore::OAuthUser,
  action in
    [AgentCore::Action::"{new_target_id}",
     AgentCore::Action::"{new_target_id}___process_number"],
  resource ==
    AgentCore::Gateway::"arn:aws:bedrock-agentcore:us-east-1:{account_id}:gateway/{new_gateway_id}"
);"""

new_policy_name = f'oauth_user_policy_int_target_{TIMESTAMP}'

new_policy_response = agentcore.create_policy(
    policyEngineId=new_policy_engine_id,
    name=new_policy_name,
    definition={'cedar': {'statement': cedar_policy_int}}
)
new_policy_id = new_policy_response['policyId']
print("✓ Cedar permit policy created for new gateway")

print("\n✓ Advanced infrastructure setup complete")

✓ New policy engine created
Waiting for policy engine to be ready...


✓ New gateway created
Waiting for gateway to be ready...


✓ Gateway target created for new gateway


✓ Cedar permit policy created for new gateway

✓ Advanced infrastructure setup complete


### 9.3 Test Gateway with Integer Parameters

Test the new gateway with the integer-based tool to verify parameter handling.

In [17]:
# Ensure URL has proper MCP endpoint
test_gateway_url = new_gateway_url if new_gateway_url.endswith('/mcp') else f"{new_gateway_url}/mcp"

# Get fresh authentication token
auth_response = cognito.admin_initiate_auth(
    UserPoolId=user_pool_id,
    ClientId=client_id,
    AuthFlow='ADMIN_NO_SRP_AUTH',
    AuthParameters={
        'USERNAME': test_username,
        'PASSWORD': 'TestPassword123!'
    }
)

id_token = auth_response['AuthenticationResult']['IdToken']

# Prepare headers
headers = {
    'Authorization': f'Bearer {id_token}',
    'Content-Type': 'application/json'
}

# Test payload with integer parameter
test_payload = {
    "jsonrpc": "2.0",
    "method": "tools/call",
    "params": {
        "name": f"{new_target_id}___process_number",
        "arguments": {
            "count": 42,
            "message": "Testing with an integer value"
        }
    },
    "id": 2
}

print("Testing gateway with integer parameter (count=42)...")
response = requests.post(
    test_gateway_url,
    json=test_payload,
    headers=headers,
    timeout=30
)

if response.status_code == 200:
    print("✓ Gateway invocation with integer parameter successful!")
else:
    print(f"✗ Failed with status {response.status_code}")
    raise RuntimeError(f"Gateway test failed: {response.text}")

Testing gateway with integer parameter (count=42)...


✓ Gateway invocation with integer parameter successful!


## Section 10: Conditional Access Control with Forbid Policies

### 10.1 Create Forbid Policy

Demonstrate advanced policy control by creating a forbid policy that denies access when the count parameter is >= 100.

In [18]:
# Create a forbid policy that denies access when count >= 100
forbid_policy = f"""forbid (
  principal,
  action == AgentCore::Action::"{new_target_id}___process_number",
  resource == AgentCore::Gateway::"arn:aws:bedrock-agentcore:us-east-1:{account_id}:gateway/{new_gateway_id}"
)
when {{ !(((context.input).count) < 100) }};"""

forbid_policy_name = f'forbid_policy_count_limit_{TIMESTAMP}'

forbid_policy_response = agentcore.create_policy(
    policyEngineId=new_policy_engine_id,
    name=forbid_policy_name,
    definition={'cedar': {'statement': forbid_policy}}
)
forbid_policy_id = forbid_policy_response['policyId']
print("✓ Forbid policy created")
print("  Rule: Deny access when count >= 100")

✓ Forbid policy created
  Rule: Deny access when count >= 100


### 10.2 Test Forbid Policy with Various Values

Test the forbid policy by invoking the tool with different count values to demonstrate conditional access control.

## Cleanup Instructions

To avoid unnecessary AWS charges, delete the resources created by this notebook:

```bash
# Delete gateways
aws bedrock-agentcore delete-gateway --gateway-identifier <gateway-id>

# Delete policy engines
aws bedrock-agentcore delete-policy-engine --policy-engine-id <policy-engine-id>

# Delete Lambda function
aws lambda delete-function --function-name gateway-target-function-<timestamp>

# Delete IAM roles and policies
aws iam delete-role-policy --role-name gateway-execution-role-<timestamp> --policy-name ComprehensiveGatewayPermissions
aws iam delete-role-policy --role-name gateway-execution-role-<timestamp> --policy-name GatewayLambdaInvokePolicy
aws iam delete-role --role-name gateway-execution-role-<timestamp>

aws iam detach-role-policy --role-name lambda-gateway-target-role-<timestamp> --policy-arn arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole
aws iam delete-role --role-name lambda-gateway-target-role-<timestamp>

# Delete Cognito User Pool
aws cognito-idp delete-user-pool --user-pool-id <user-pool-id>
```

Replace `<timestamp>`, `<gateway-id>`, `<policy-engine-id>`, and `<user-pool-id>` with actual values from the notebook execution.

## Section 10: Conditional Access Control with Forbid Policies

### 10.1 Create Forbid Policy

Demonstrate advanced policy control by creating a forbid policy that denies access when the count parameter is >= 100.

In [ ]:
# Create a forbid policy that denies access when count >= 100
forbid_policy = f"""forbid (
  principal,
  action == AgentCore::Action::"{new_target_id}___process_number",
  resource == AgentCore::Gateway::"arn:aws:bedrock-agentcore:us-east-1:{account_id}:gateway/{new_gateway_id}"
)
when {{ !(((context.input).count) < 100) }};"""

forbid_policy_name = f'forbid_policy_count_limit_{TIMESTAMP}'

forbid_policy_response = agentcore.create_policy(
    policyEngineId=new_policy_engine_id,
    name=forbid_policy_name,
    definition={'cedar': {'statement': forbid_policy}}
)
forbid_policy_id = forbid_policy_response['policyId']
print("✓ Forbid policy created")
print("  Rule: Deny access when count >= 100")

### 10.2 Test Forbid Policy with Various Values

Test the forbid policy by invoking the tool with different count values to demonstrate conditional access control.

In [20]:
# Test cases: below limit, at boundary, above limit
test_cases = [
    {"count": 50, "message": "Below limit (count=50)", "expected": "ALLOWED"},
    {"count": 99, "message": "At boundary (count=99)", "expected": "ALLOWED"},
    {"count": 100, "message": "At limit (count=100)", "expected": "DENIED"},
    {"count": 150, "message": "Above limit (count=150)", "expected": "DENIED"}
]

# Get fresh authentication token
auth_response = cognito.admin_initiate_auth(
    UserPoolId=user_pool_id,
    ClientId=client_id,
    AuthFlow='ADMIN_NO_SRP_AUTH',
    AuthParameters={
        'USERNAME': test_username,
        'PASSWORD': 'TestPassword123!'
    }
)

id_token = auth_response['AuthenticationResult']['IdToken']
headers = {
    'Authorization': f'Bearer {id_token}',
    'Content-Type': 'application/json'
}

print("\n" + "="*80)
print("FORBID POLICY TEST: Conditional Access Control")
print("="*80)
print("Policy Rule: Deny access when count >= 100\n")

results = []
for i, test_case in enumerate(test_cases, 1):
    count = test_case['count']
    message = test_case['message']
    expected = test_case['expected']
    
    test_payload = {
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": f"{new_target_id}___process_number",
            "arguments": {
                "count": count,
                "message": message
            }
        },
        "id": i + 100
    }
    
    # Make the request
    response = requests.post(
        test_gateway_url,
        json=test_payload,
        headers=headers,
        timeout=30
    )
    
    # Parse response
    response_data = response.json()
    
    # Check for error in response
    if 'error' in response_data:
        actual = "DENIED"
    else:
        actual = "ALLOWED"
    
    # Check if result matches expected
    match = "✓" if actual == expected else "✗"
    
    results.append({
        "count": count,
        "message": message,
        "expected": expected,
        "actual": actual,
        "match": match
    })
    
    print(f"Test {i}: {message:30} | Expected: {expected:7} | Actual: {actual:7} | {match}")
    print(f"         Response: {response_data}")

print("="*80)
print("SUMMARY")
print("="*80)
print("✓ count < 100  → ALLOWED (permit policy permits, forbid doesn't apply)")
print("✗ count >= 100 → DENIED  (forbid policy blocks the request in response body)")
print("="*80)
print("\nThis demonstrates parameter-based conditional access control using Cedar policies.")
print("="*80)


FORBID POLICY TEST: Conditional Access Control
Policy Rule: Deny access when count >= 100



Test 1: Below limit (count=50)         | Expected: ALLOWED | Actual: ALLOWED | ✓
         Response: {'jsonrpc': '2.0', 'id': 101, 'result': {'isError': False, 'content': [{'type': 'text', 'text': '{"statusCode":200,"body":{"message":"Gateway test successful","event":{"count":50,"message":"Below limit (count=50)"}}}'}]}}


Test 2: At boundary (count=99)         | Expected: ALLOWED | Actual: ALLOWED | ✓
         Response: {'jsonrpc': '2.0', 'id': 102, 'result': {'isError': False, 'content': [{'type': 'text', 'text': '{"statusCode":200,"body":{"message":"Gateway test successful","event":{"count":99,"message":"At boundary (count=99)"}}}'}]}}


Test 3: At limit (count=100)           | Expected: DENIED  | Actual: DENIED  | ✓
         Response: {'jsonrpc': '2.0', 'id': 103, 'error': {'code': -32002, 'message': 'Tool Execution Denied: Tool call not allowed due to policy enforcement [Policy evaluation denied due to forbid_policy_count_limit_05032026172451-0en8klxj_7]'}}


Test 4: Above limit (count=150)        | Expected: DENIED  | Actual: DENIED  | ✓
         Response: {'jsonrpc': '2.0', 'id': 104, 'error': {'code': -32002, 'message': 'Tool Execution Denied: Tool call not allowed due to policy enforcement [Policy evaluation denied due to forbid_policy_count_limit_05032026172451-0en8klxj_7]'}}
SUMMARY
✓ count < 100  → ALLOWED (permit policy permits, forbid doesn't apply)
✗ count >= 100 → DENIED  (forbid policy blocks the request in response body)

This demonstrates parameter-based conditional access control using Cedar policies.


## Cleanup Instructions

To avoid unnecessary AWS charges, delete the resources created by this notebook:

```bash
# Delete gateways
aws bedrock-agentcore delete-gateway --gateway-identifier <gateway-id>

# Delete policy engines
aws bedrock-agentcore delete-policy-engine --policy-engine-id <policy-engine-id>

# Delete Lambda function
aws lambda delete-function --function-name gateway-target-function-<timestamp>

# Delete IAM roles and policies
aws iam delete-role-policy --role-name gateway-execution-role-<timestamp> --policy-name ComprehensiveGatewayPermissions
aws iam delete-role-policy --role-name gateway-execution-role-<timestamp> --policy-name GatewayLambdaInvokePolicy
aws iam delete-role --role-name gateway-execution-role-<timestamp>

aws iam detach-role-policy --role-name lambda-gateway-target-role-<timestamp> --policy-arn arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole
aws iam delete-role --role-name lambda-gateway-target-role-<timestamp>

# Delete Cognito User Pool
aws cognito-idp delete-user-pool --user-pool-id <user-pool-id>
```

Replace `<timestamp>`, `<gateway-id>`, `<policy-engine-id>`, and `<user-pool-id>` with actual values from the notebook execution.